# Lesson 1: Your First LLM Call

## WHY
Every agentic bot starts with one skill: asking an LLM a question and getting an answer back.  
In production, you need to control **which model** you call, **how creative** the response is, and **how many tokens** you spend (because tokens = money).  
LangChain's `ChatOpenAI` class gives you a single, consistent interface to do all of that.

**By the end of this notebook you will:**
1. Create a `ChatOpenAI` instance with explicit model parameters
2. Understand the three message roles: `SystemMessage`, `HumanMessage`, `AIMessage`
3. Call `.invoke()` and inspect the response object
4. Experiment with `temperature` and `max_tokens`
5. Stream a response token-by-token
6. Package everything into importable helper functions

## Setup
We load your OpenAI API key from the `.env` file at the project root.  
`dotenv` reads it into `os.environ` so LangChain picks it up automatically — you never pass the key directly.

In [ ]:
import os
from dotenv import load_dotenv

# load_dotenv() walks up directories until it finds a .env file
load_dotenv()

# Quick sanity check — should print True if your .env has OPENAI_API_KEY set
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## WHAT — `ChatOpenAI`

`ChatOpenAI` is LangChain's wrapper around the OpenAI chat completions API.  
Key parameters you'll use constantly:

| Parameter | What it does | Typical value |
|-----------|-------------|---------------|
| `model` | Which OpenAI model to call | `"gpt-4o-mini"` (cheap & fast) or `"gpt-4o"` |
| `temperature` | Randomness: 0 = deterministic, 2 = wild | `0` for moderation, `0.7` for creative |
| `max_tokens` | Hard cap on response length | `256` for short answers |

📖 [ChatOpenAI API reference](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)

In [ ]:
from langchain_openai import ChatOpenAI

# Create a chat model instance
# - model: gpt-4o-mini is the cheapest, fastest option for learning
# - temperature: 0 means deterministic (same input → same output)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

print(type(llm))
print(f"Model: {llm.model_name}, Temperature: {llm.temperature}")

## WHAT — Message Types

Chat models don't take raw strings. They take a **list of messages**, each with a role:

| Class | Role | Purpose |
|-------|------|---------|
| `SystemMessage` | `system` | Sets the LLM's persona / instructions — the LLM always "obeys" this |
| `HumanMessage` | `user` | What the user said — this is the input you're responding to |
| `AIMessage` | `assistant` | The LLM's own previous responses (used for multi-turn conversations) |

For our moderation bot, the **system message** will eventually contain the moderation rules.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# Build a message list — this is what the LLM actually sees
messages = [
    SystemMessage(content="You are a helpful Discord bot assistant."),
    HumanMessage(content="What can you help me with?"),
]

# .invoke() sends the messages to OpenAI and returns an AIMessage
response = llm.invoke(messages)

print(response.content)

## HOW — Experimenting with Model Parameters

Let's see how `temperature` and `max_tokens` change the output.  
Run this cell a few times — with `temperature=0` the output is identical every time.  
Then change it to `1.0` and watch the variation.

In [ ]:
# Try different parameter combinations
creative_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=1.0,   # high creativity
    max_tokens=100,     # short response
)

response = creative_llm.invoke([
    SystemMessage(content="You are a pirate. Speak like one."),
    HumanMessage(content="Tell me about Python programming."),
])

print(response.content)
print(f"\n--- Stopped at max_tokens? finish_reason: {response.response_metadata['finish_reason']}")

## WHAT — The Response Object (`AIMessage`)

`.invoke()` doesn't return a string — it returns an `AIMessage` object packed with metadata.  
The important attributes:

| Attribute | What it tells you |
|-----------|------------------|
| `.content` | The actual text response |
| `.response_metadata` | Model name, finish reason, token counts |
| `.usage_metadata` | Input/output/total token counts (for cost tracking) |
| `.id` | Unique ID for this completion |

In [ ]:
# Make a call and inspect everything
response = llm.invoke([
    SystemMessage(content="You are a content moderator."),
    HumanMessage(content="Is this message appropriate: 'Hello everyone!'"),
])

print("=== Content ===")
print(response.content)

print("\n=== Token Usage ===")
print(f"Input tokens:  {response.usage_metadata['input_tokens']}")
print(f"Output tokens: {response.usage_metadata['output_tokens']}")
print(f"Total tokens:  {response.usage_metadata['total_tokens']}")

print("\n=== Metadata ===")
print(f"Model: {response.response_metadata['model_name']}")
print(f"Finish reason: {response.response_metadata['finish_reason']}")

## HOW — Streaming Responses

For a Discord bot, streaming lets you show a "typing..." indicator and update the message as tokens arrive.  
`.stream()` returns an iterator of `AIMessageChunk` objects — each chunk has a `.content` property with the next piece of text.

In [ ]:
# .stream() yields AIMessageChunk objects one at a time
for chunk in llm.stream([
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Explain what a Discord bot is in 2 sentences."),
]):
    print(chunk.content, end="", flush=True)

print()  # newline at the end

## Importable Helpers

Below are two functions written so you can copy them straight into the bot codebase later.  
They follow the project's Google/NumPy docstring convention and return clean types.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


def create_chat_model(
    model: str = "gpt-4o-mini",
    temperature: float = 0,
    max_tokens: int | None = None,
) -> ChatOpenAI:
    """Create a configured ChatOpenAI instance.

    Parameters
    ----------
    model : str
        OpenAI model name (e.g. "gpt-4o-mini", "gpt-4o").
    temperature : float
        Sampling temperature. 0 = deterministic.
    max_tokens : int | None
        Maximum tokens in the response. None = no limit.

    Returns
    -------
    ChatOpenAI
        A ready-to-use chat model instance.
    """
    return ChatOpenAI(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
    )


def get_llm_response(
    user_input: str,
    system_prompt: str = "You are a helpful assistant.",
    llm: ChatOpenAI | None = None,
) -> str:
    """Send a message to the LLM and return the text response.

    Parameters
    ----------
    user_input : str
        The user's message.
    system_prompt : str
        Instructions for the LLM's behavior.
    llm : ChatOpenAI | None
        Chat model instance. Creates a default one if None.

    Returns
    -------
    str
        The LLM's text response.
    """
    if llm is None:
        llm = create_chat_model()

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_input),
    ]
    response = llm.invoke(messages)
    return response.content

In [ ]:
# Quick test of the helper functions
llm = create_chat_model(model="gpt-4o-mini", temperature=0)

answer = get_llm_response(
    user_input="Should I ban someone for saying 'hello'?",
    system_prompt="You are a Discord server moderator. Be concise.",
    llm=llm,
)

print(answer)

## Summary

| Concept | Key takeaway |
|---------|-------------|
| `ChatOpenAI` | LangChain's wrapper — pass `model`, `temperature`, `max_tokens` |
| Messages | Always a list: `[SystemMessage, HumanMessage, ...]` |
| `.invoke()` | Sends messages, returns `AIMessage` with `.content` and `.usage_metadata` |
| `.stream()` | Returns chunks for real-time display |
| `temperature=0` | Deterministic — essential for moderation |

**Next up → Lesson 2:** LCEL chains, prompt templates, few-shot examples, and swapping to Anthropic.